In [4]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas(desc="Progress")

df = pd.read_csv("Veritasium_comments_NER-KEY_2.csv")
print(df.shape)
df.head()

(12899, 11)


,comment_id,author,updated_at,like_count,text,text_normalized,tokens_str,entities_str,entity_texts_str,entity_types_str,keywords_str
0,0,@DanielRenardAnimation,2019-09-19T18:28:21Z,5751,*Russian Cosmonaut spins a wingnut in space:* ...,russian cosmonaut spins a wingnut in space tel...,russian cosmonaut spins wingnut space tell one,"[('russian', 'NORP')]",russian,NORP,"cosmonaut, russian, tell, wingnut, spins"
1,1,@NLTops,2020-03-07T07:17:39Z,5244,Someone should tell the Flat Earthers of the D...,someone should tell the flat earthers of the d...,someone tell flat earthers dzhanibekov effect ...,[],NaN,NaN,"ll, sooner, world flip, flat earthers, earthers"
2,2,@aviatordude1961,2021-07-26T02:06:48Z,4005,I thought the reason the Russians kept this a ...,i thought the reason the russians kept this a ...,thought reason russians kept secret going fema...,"[('russians', 'NORP')]",russians,NORP,"win, would always, gold, russians kept, gymnasts"
3,3,@alvirahesc7436,2021-04-10T12:21:42Z,3842,"""Babe, come over, im home alone""\n\n""No, babe,...","babe, come over, im home alone no, babe, im so...",babe come home alone babe solvin centuries old...,"[('centuries', 'DATE')]",centuries,DATE,"old math, home, centuries, alone, old"
4,4,@zeekjones1,2019-09-20T05:46:15Z,2814,Over 300 people broke their phone after watchi...,over 300 people broke their phone after watchi...,300 people broke phone watching video,"[('over 300', 'CARDINAL')]",over 300,CARDINAL,"300, phone watching, broke, watching video, wa..."


VADER sentiment (rule-based, fast).

In [5]:
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    scores = analyzer.polarity_scores(str(text))
    compound = scores['compound']
    if compound >= 0.05:
        label = 'positive'
    elif compound <= -0.05:
        label = 'negative'
    else:
        label = 'neutral'
    return pd.Series([compound, label])

df[['vader_score', 'vader_label']] = df['text_normalized'].progress_apply(get_vader_sentiment)

df[['text_normalized','vader_score','vader_label']].head(10)

Progress: 100%|██████████| 12899/12899 [00:02<00:00, 4780.98it/s]


,text_normalized,vader_score,vader_label
0,russian cosmonaut spins a wingnut in space tel...,-0.3595,negative
1,someone should tell the flat earthers of the d...,0.2023,positive
2,i thought the reason the russians kept this a ...,0.6239,positive
3,"babe, come over, im home alone no, babe, im so...",-0.5719,negative
4,over 300 people broke their phone after watchi...,-0.4215,negative
5,oh! so thats why my liquid filled bullets keep...,0.0000,neutral
6,this explaination is beautiful when you're act...,0.8118,positive
7,750 australians didnt like the fact earth wont...,-0.2755,negative
8,video contains the phrase prove feynman wrong ...,-0.4767,negative
9,"as a kid, i would frequently watch my dad flip...",0.7003,positive


TweetNLP Sentiment (transformer-based, more accurate but slower).

In [6]:
from transformers import pipeline

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512
)

def get_tweetnlp_sentiment(text):
    try:
        result = sentiment_pipe(str(text))[0]
        label_map = {'LABEL_0':'negative','LABEL_1':'neutral','LABEL_2':'positive',
                      'negative':'negative','neutral':'neutral','positive':'positive'}
        return pd.Series([label_map.get(result['label'], result['label']), result['score']])
    except Exception:
        return pd.Series(['neutral', 0.0])

df[['tweetnlp_label','tweetnlp_score']] = df['text_normalized'].progress_apply(get_tweetnlp_sentiment)

df[['text_normalized','vader_label','tweetnlp_label']].head(10)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Progress: 100%|██████████| 12899/12899 [29:09<00:00,  7.37it/s]


,text_normalized,vader_label,tweetnlp_label
0,russian cosmonaut spins a wingnut in space tel...,negative,negative
1,someone should tell the flat earthers of the d...,positive,negative
2,i thought the reason the russians kept this a ...,positive,neutral
3,"babe, come over, im home alone no, babe, im so...",negative,neutral
4,over 300 people broke their phone after watchi...,negative,negative
5,oh! so thats why my liquid filled bullets keep...,neutral,negative
6,this explaination is beautiful when you're act...,positive,positive
7,750 australians didnt like the fact earth wont...,negative,negative
8,video contains the phrase prove feynman wrong ...,negative,neutral
9,"as a kid, i would frequently watch my dad flip...",positive,neutral


Compare and save.

In [7]:
print(df['vader_label'].value_counts())
print(df['tweetnlp_label'].value_counts())

df.to_csv("Veritasium_comments_sentiment.csv", index=False)
print(f"Saved {len(df)} comments with sentiment")

vader_label
positive    5754
neutral     4621
negative    2524
Name: count, dtype: int64
tweetnlp_label
neutral     8110
negative    2576
positive    2213
Name: count, dtype: int64
Saved 12899 comments with sentiment
